# Phase 7: Advanced Modeling - Dubai Apartment Price Prediction

## Objectives
- Load the cleaned dataset `ready_cleaned_v2.csv`
- Train advanced gradient boosting models: XGBoost, LightGBM, CatBoost
- Use 5-fold cross-validation to compare model performance
- Optimize hyperparameters for the best performing model using Optuna
- Save the best model and preprocessor pipeline to disk using `joblib` for deployment

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
print('Advanced modeling imports successful!')

Advanced modeling imports successful!


C:\Users\adith\OneDrive\Desktop\Real-estate\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load clean dataset

In [2]:
df = pd.read_csv('data/ready_cleaned_v2.csv')
print('Cleaned dataset dimensions:', df.shape)

Cleaned dataset dimensions: (16105, 102)


### Define Preprocessor Pipeline

In [3]:
num_features = ['beds', 'baths', 'area', 'luxury_score', 'has_view', 'has_maids_room', 'is_freehold', 'dist_to_burj', 'dist_to_airport', 'dist_to_beach']
cat_features = ['furnished']
target = 'log_price'

X = df[num_features + ['district'] + cat_features]
y = df[target]

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import KFold

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('district_te', TargetEncoder(smooth='auto', cv=KFold(n_splits=5, shuffle=True, random_state=42)), ['district']),
    ('cat', cat_transformer, cat_features)
])

X_proc = preprocessor.fit_transform(X, y)
print('Features preprocessed successfully!')

Features preprocessed successfully!


### Compare Gradient Boosting Models using 5-Fold Cross-Validation
We evaluate XGBoost, LightGBM, and CatBoost Regressors. Metrics are evaluated on the original price scale (AED).

In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
models = {
    'XGBoost': XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
    'LightGBM': LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
    'CatBoost': CatBoostRegressor(random_state=42, verbose=0)
}

model_comparison = []
for name, model in models.items():
    maes = []
    rmses = []
    r2s = []
    
    for train_idx, val_idx in kf.split(X_proc):
        # Split
        X_tr, X_va = X_proc[train_idx], X_proc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
        
        # Fit and predict
        model.fit(X_tr, y_tr)
        preds_log = model.predict(X_va)
        preds = np.expm1(preds_log)
        y_va_orig = np.expm1(y_va)
        
        maes.append(mean_absolute_error(y_va_orig, preds))
        rmses.append(np.sqrt(mean_squared_error(y_va_orig, preds)))
        r2s.append(r2_score(y_va_orig, preds))
        
    model_comparison.append({
        'Model': name,
        'Mean MAE (AED)': np.mean(maes),
        'Mean RMSE (AED)': np.mean(rmses),
        'Mean R2': np.mean(r2s)
    })
    print(f'{name} -> Mean MAE: {np.mean(maes):,.2f} AED, Mean R2: {np.mean(r2s):.4f}')

comparison_df = pd.DataFrame(model_comparison)
comparison_df

XGBoost -> Mean MAE: 841,254.15 AED, Mean R2: 0.5072


C:\Users\adith\OneDrive\Desktop\Real-estate\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\adith\OneDrive\Desktop\Real-estate\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


C:\Users\adith\OneDrive\Desktop\Real-estate\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\adith\OneDrive\Desktop\Real-estate\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


C:\Users\adith\OneDrive\Desktop\Real-estate\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LightGBM -> Mean MAE: 819,265.27 AED, Mean R2: 0.5316


CatBoost -> Mean MAE: 820,557.22 AED, Mean R2: 0.5264


,Model,Mean MAE (AED),Mean RMSE (AED),Mean R2
0,XGBoost,841254.151923,1.420595e+06,0.507176
1,LightGBM,819265.267469,1.384927e+06,0.531638
2,CatBoost,820557.221606,1.392491e+06,0.526406


### Hyperparameter Tuning for XGBoost using Optuna

In [5]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': 0
    }
    model = XGBRegressor(**params)
    scores = []
    for train_idx, val_idx in kf.split(X_proc):
        X_tr, X_va = X_proc[train_idx], X_proc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
        model.fit(X_tr, y_tr)
        preds = np.expm1(model.predict(X_va))
        scores.append(mean_absolute_error(np.expm1(y_va), preds))
    return np.mean(scores)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10)
print('Best trial MAE:', study.best_value)
print('Best trial parameters:', study.best_params)

Best trial MAE: 816954.3058405776
Best trial parameters: {'n_estimators': 400, 'max_depth': 7, 'learning_rate': 0.016707875184726384, 'subsample': 0.6748504505221091, 'colsample_bytree': 0.9998317000826489}


### Train Final Tuned Model and Save Artifacts

In [6]:
# Train model on full dataset
best_xgb_params = study.best_params
xgb_tuned = XGBRegressor(**best_xgb_params, random_state=42, n_jobs=-1, verbosity=0)

lgb_tuned = LGBMRegressor(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

cat_tuned = CatBoostRegressor(
    iterations=600,
    depth=6,
    learning_rate=0.05,
    random_state=42,
    verbose=0
)

from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
final_ensemble = StackingRegressor(
    estimators=[
        ('xgb', xgb_tuned),
        ('lgb', lgb_tuned),
        ('cat', cat_tuned)
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1
)

final_ensemble.fit(X_proc, y)

# Create directory if missing
os.makedirs('models', exist_ok=True)

# Serialize preprocessor and ensemble model
joblib.dump(final_ensemble, 'models/model.pkl')
joblib.dump(preprocessor, 'models/preprocessor.pkl')
print('Tuned StackingRegressor Ensemble (with Ridge meta-model) and preprocessor saved to models/ directory successfully!')

Tuned StackingRegressor Ensemble (with Ridge meta-model) and preprocessor saved to models/ directory successfully!
